https://archive.ics.uci.edu/dataset/211/communities+and+crime+unnormalized

In [10]:
#%pip install ucimlrepo
#%pip install seaborn

In [11]:
from ucimlrepo import fetch_ucirepo 

import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [12]:
# Carrega o dataset
communities_and_crime_unnormalized = fetch_ucirepo(id=211) 

# X recebe as variáveis de entrada (features)
X = communities_and_crime_unnormalized.data.features 

# y recebe as variáveis alvo (targets)
y = communities_and_crime_unnormalized.data.targets

In [13]:
# Concatena as features (X) e os alvos (y) em um único DataFrame
df = pd.concat([X, y], axis=1)

In [14]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print("total missing:", df.isnull().sum().sum())
print(df.isnull().sum().sort_values(ascending=False).head(30))

shape: (2215, 143)
columns: ['State', 'pop', 'perHoush', 'pctBlack', 'pctWhite', 'pctAsian', 'pctHisp', 'pct12-21', 'pct12-29', 'pct16-24', 'pct65up', 'persUrban', 'pctUrban', 'medIncome', 'pctWwage', 'pctWfarm', 'pctWdiv', 'pctWsocsec', 'pctPubAsst', 'pctRetire', 'medFamIncome', 'perCapInc', 'whitePerCap', 'blackPerCap', 'NAperCap', 'asianPerCap', 'otherPerCap', 'hispPerCap', 'persPoverty', 'pctPoverty', 'pctLowEdu', 'pctNotHSgrad', 'pctCollGrad', 'pctUnemploy', 'pctEmploy', 'pctEmployMfg', 'pctEmployProfServ', 'pctOccupManu', 'pctOccupMgmt', 'pctMaleDivorc', 'pctMaleNevMar', 'pctFemDivorc', 'pctAllDivorc', 'persPerFam', 'pct2Par', 'pctKids2Par', 'pctKids-4w2Par', 'pct12-17w2Par', 'pctWorkMom-6', 'pctWorkMom-18', 'kidsBornNevrMarr', 'pctKidsBornNevrMarr', 'numForeignBorn', 'pctFgnImmig-3', 'pctFgnImmig-5', 'pctFgnImmig-8', 'pctFgnImmig-10', 'pctImmig-3', 'pctImmig-5', 'pctImmig-8', 'pctImmig-10', 'pctSpeakOnlyEng', 'pctNotSpeakEng', 'pctLargHousFam', 'pctLargHous', 'persPerOccupHous',

In [15]:
# Obtém número de linhas e colunas do DataFrame
n_rows, n_cols = df.shape

# Conta valores nulos por coluna
missing_sum = df.isnull().sum()

# Filtra apenas colunas que possuem pelo menos 1 valor faltante
cols_with_missing = missing_sum[missing_sum > 0]

# Imprime total geral de valores faltantes no dataset
print(f"Total de valores faltantes: {missing_sum.sum()}")

# Imprime quantidade de colunas que possuem valores faltantes
print(f"Colunas afetadas: {len(cols_with_missing)}")

# Calcula percentual de nulos por coluna e mostra as 5 com maior proporção
print(f"colunas com mais nulos (%):")
print((cols_with_missing / n_rows * 100).sort_values(ascending=False).head())

Total de valores faltantes: 42147
Colunas afetadas: 39
colunas com mais nulos (%):
numPolice            84.514673
policePerPop         84.514673
policeField          84.514673
policeCalls          84.514673
policeFieldPerPop    84.514673
dtype: float64


Colunas com mais de 80% de valores faltantes foram removidas por apresentarem informação insuficiente para imputação confiável, reduzindo risco de viés e overfitting. Colunas com menos de 10% de ausência foram mantidas e imputadas via mediana.

In [16]:
# colunas com >80% de nulos
missing_cols = df.columns[df.isnull().mean() > 0.8]

# dropar essas colunas
df = df.drop(columns=missing_cols)

# conferir novo shape
print(df.shape)

(2215, 121)


In [17]:
print(df.isnull().sum().sort_values(ascending=False).head(10))

violentPerPop    221
rapes            208
rapesPerPop      208
nonViolPerPop     97
arsonsPerPop      91
arsons            91
assaultPerPop     13
assaults          13
larcPerPop         3
autoTheft          3
dtype: int64


Imputação da mediana no resto dos dados faltante

In [ ]:
threshold = X['ViolentPerPop'].median()

df['violent_class'] = (df['ViolentPerPop'] > threshold).astype(int)

print("Threshold:", threshold)
print(df['violent_class'].value_counts(normalize=True))

KeyError: 'ViolentPerPop'

In [ ]:
crime_cols = [
    'murders','rapes','robberies','assaults','burglaries',
    'larcenies','autoTheft','arsons',
    'autoTheftPerPop','arsonsPerPop',
    'murdPerPop','rapesPerPop','robbbPerPop',
    'assaultPerPop','burglPerPop','larcPerPop',
    'ViolentCrimesPerPop','nonViolPerPop'
]

df = df.drop(columns=[c for c in crime_cols if c in df.columns])

print("Novo shape:", df.shape)

In [ ]:
X = df.drop(columns=['violent_class'])
y = df['violent_class']

print(X.shape, y.shape)

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = X.select_dtypes(include=['number']).columns

imputer = SimpleImputer(strategy='median')
X[num_cols] = imputer.fit_transform(X[num_cols])

print("Missing após imputação:", X.isnull().sum().sum())

NAs restantes: 0


In [ ]:
y = pd.Series(y, name='violent_class')
df = pd.concat([X, y], axis=1)

In [ ]:
# remover linhas onde o target é nulo
df = df.dropna(subset=['violent_class'])

# conferir
print("Novo shape:", df.shape)
print("NAs no target:", df['violent_class'].isnull().sum())

Novo shape: (2215, 106)
NAs no target: 0


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
df.head()

,State,pop,perHoush,pctBlack,pctWhite,pctAsian,pctHisp,pct12-21,pct12-29,pct16-24,...,pctSameHouse-5,pctSameCounty-5,pctSameState-5,landArea,popDensity,pctUsePubTrans,pctOfficDrugUnit,violentPerPop,nonViolPerPop,violent_class
0,NJ,11980.0,3.10,1.37,91.78,6.50,1.88,12.47,21.44,10.93,...,65.29,78.09,89.14,6.5,1845.9,9.63,0.0,41.02,1394.59,0
1,PA,23123.0,2.82,0.80,95.57,3.44,0.85,11.01,21.30,10.48,...,71.27,90.22,96.12,10.6,2186.7,3.84,0.0,127.56,1955.95,0
2,OR,29344.0,2.43,0.74,94.33,3.43,2.35,11.36,25.88,11.01,...,36.60,61.26,82.85,10.6,2780.9,4.37,0.0,218.59,6167.51,0
3,NY,16656.0,2.40,1.70,97.35,0.50,0.70,12.55,25.20,12.19,...,56.70,90.17,96.24,5.2,3217.7,3.31,0.0,306.64,4425.45,0
4,MN,11245.0,2.76,0.53,89.16,1.17,0.52,24.46,40.53,28.69,...,42.22,60.34,89.02,11.5,974.2,0.38,0.0,374.06,9988.79,0


In [ ]:
non_predictive = [
    'State'
]

df = df.drop(columns=[c for c in non_predictive if c in df.columns])

print("Novo shape:", df.shape)

Novo shape: (2215, 105)


In [ ]:
# separar X e y
X = df.drop(columns=['violent_class'])
y = df['violent_class']

# split estratificado
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

svm = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[283  50]
 [141 191]]
              precision    recall  f1-score   support

           0       0.67      0.85      0.75       333
           1       0.79      0.58      0.67       332

    accuracy                           0.71       665
   macro avg       0.73      0.71      0.71       665
weighted avg       0.73      0.71      0.71       665



In [ ]:
svm = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm.fit(X_train, y_train)

probs = svm.predict_proba(X_test)[:,1]

y_pred_adj = (probs > 0.4).astype(int)  # exemplo threshold menor

print(classification_report(y_test, y_pred_adj))